# 01 - Collect a Training Dataset

This notebook shows the standard Mouse Core data collection pattern:

1. Build one or more environments with `mouse_gym.EnvConfig` and `make_group_env`.
1. Create one `Datastore` for each environment stream.
1. Record the coordinator metadata and any selected action, step the grouped
   environment, and record each result.
1. Publish the stores with `push_stores_to_hub` so training notebooks can load the same
   data later.

The environment used here is Procedural Frozen Lake wrapped by `RepeatTaskEnv`, then by
mouse-gym. Any Gymnasium env can take that same path: `env_fn` is the mouse-gym stream,
and a `Datastore` stores the event records.

This is a short usage example. Increase the collection limits to build a larger training
dataset.

Payload fields intended only for supervision or episode control start with `_`:
`_q_star`, `_episode_start`, `_terminated`, and `_truncated`. This is a naming
convention only. Tokenizer field specifications explicitly choose what becomes model
input; objective specifications map these payload names to the usual loss names.


In [ ]:
import gymnasium as gym
import numpy as np
import torch
from mouse_gym import EnvConfig, RepeatTaskTracker, make_group_env
from procedural_frozenlake import ensure_registered
from repeat_task_gym import RepeatTaskEnv

from mouse_core.data import EventLog, push_stores_to_hub

# Repositories
# Hugging Face dataset repo for push_stores_to_hub
DATASET_ID = "mouse-example-frozenlake-dataset"

# Environment
# max steps per episode
MAX_STEPS_PER_EPISODE = 24
# inner episodes per outer task
REPEAT_COUNT = 20
# number of environment streams in the GroupEnv
NUM_ENVS = 4
# probability of choosing the oracle action; otherwise sample randomly
ORACLE_PROB = 0.5

# Collection and rollout
STEPS_PER_ENV = 10000

# Runtime
ensure_registered()

## Build Environment

`EnvConfig` builds one mouse-gym stream. Any Gymnasium env can be that stream: pass it
through `env_fn`. This notebook wraps Procedural FrozenLake in `RepeatTaskEnv` first, so
several lake episodes are one mouse-gym episode.

- `name` is the stream name reused for the matching `Datastore`.
- `seed` plus `seed_mode="stream"` gives each mouse-gym reset a new seed. That reset is
  an episode start. `reset_options={"regenerate_map": True}` builds a new lake there.
- `RepeatTaskEnv` keeps the inner episodes. `max_task_episodes` ends the episode, which
  mouse-gym reports as `truncated`. `episode_seed_mode="per_task"` and
  `episode_reset_options={"regenerate_map": False}` reuse that lake for every inner
  repeat of the outer episode.
- `q_star_horizon` matches `max_episode_steps`, so oracle values account for the
  remaining actions in each inner episode.
- The collector records mouse-gym `terminated` and `truncated` as `_terminated` and
  `_truncated` in the payload. Inner episode ends stay on the observation
  (`episode_terminated`, `episode_truncated`) and are copied onto the row.


In [ ]:
def make_lake(*, stream: int):
    def env_fn():
        lake = gym.make(
            "Procedural-FrozenLake-v1",
            width=8,
            height=8,
            max_episode_steps=MAX_STEPS_PER_EPISODE,
            q_star_horizon=MAX_STEPS_PER_EPISODE,
            map_seed=stream,
            slippery_success_rate=1.0,
            permute_obs=True,
            permute_actions=True,
            render_mode=None,
            emit_q_star=True,
        )
        return RepeatTaskEnv(
            lake,
            max_task_episodes=REPEAT_COUNT,
            episode_seed_mode="per_task",
            episode_reset_options={"regenerate_map": False},
        )

    return env_fn


configs = [
    EnvConfig(
        name=f"proc_frozenlake_{i}",
        seed=i,
        seed_mode="stream",
        env_fn=make_lake(stream=i),
        reset_options={"regenerate_map": True},
    )
    for i in range(NUM_ENVS)
]

env = make_group_env(configs)

## Record Events

Keep one `EventLog` per environment stream. `log.to_store()` converts it to a
`Datastore`. Each record contains `provider`, JSON `payload`, and top-level
`episode_index` and `step_index` for sampling. The order is **coordinator → agent →
environment**. Omit the coordinator when its payload is empty, and omit the agent on
resets or passive steps.

The coordinator records episode-start markers. The agent records the selected action.
The environment records its observation, reward, ending flags, and `action_requested`
when it needs another action. All events for one output share its `step_index`; it
restarts at zero in each outer episode.

A standalone `RepeatTaskTracker` tracks the latest output and episode state for each
stream. Its zero-based `repeat_episode_index` is recorded only on inner reset
environment events, including the first reset. `env.step(...)` stays explicit. The log
only records the supplied events.


In [ ]:
logs = [EventLog(name=name) for name in env.names]
trackers = [RepeatTaskTracker() for _ in logs]

Choose and record an action only when the environment requests one. `None` lets the
wrapper reset after an episode ends. `ORACLE_PROB` is constant throughout collection.

Record `_q_star` only when an action is requested, `reward` only when nonzero, and
boolean flags only when true. `_episode_start` marks a new outer task. Repeat episode
indices are logged only on inner resets. Both indices remain on every event. Use
local `sample_ranges` and `sample_eligible` functions, as shown in notebook 02,
for whole completed tasks. A custom range function can instead select N
consecutive whole steps within a task. Indexed datasets reject ranges that split a step
or cross task boundaries.

Decoded views preserve missing fields. Optional tokenizer inputs skip them. Training
objectives explicitly configure `missing_value` for labels such as zero reward or absent
action; this does not add fields to the view or model input.


In [ ]:
if not 0.0 <= ORACLE_PROB <= 1.0:
    raise ValueError("ORACLE_PROB must be between 0 and 1")

for _ in range(STEPS_PER_ENV):
    for log, tracker in zip(logs, trackers):
        episode_start = tracker.output is None or tracker.ended
        if episode_start:
            log.event(
                provider="coordinator",
                episode_index=tracker.next_episode_index,
                step_index=(0 if episode_start else tracker.step_index + 1),
                payload={
                    "_episode_start": True,
                },
            )

    actions: list[int | None] = [None] * len(logs)
    for i, (log, tracker) in enumerate(zip(logs, trackers)):
        if tracker.action_requested:
            if torch.rand(1).item() < ORACLE_PROB:
                actions[i] = int(np.argmax(tracker.info["q_star"]))
            else:
                actions[i] = int(env.action_space.spaces[i].sample())
            log.event(
                provider="agent",
                episode_index=tracker.next_episode_index,
                step_index=tracker.step_index + 1,
                payload={
                    "action": actions[i],
                },
            )

    outputs = env.step(tuple(actions))
    for i, (tracker, output) in enumerate(zip(trackers, outputs)):
        tracker.update(output=output)
        log = logs[i]

        log.event(
            provider="environment",
            episode_index=tracker.episode_index,
            step_index=tracker.step_index,
            payload={
                "observation": tracker.observation,
                **(
                    {"repeat_episode_index": tracker.repeat_episode_index}
                    if tracker.repeat_episode_start
                    else {}
                ),
                **({"inner_terminated": True} if tracker.inner_terminated else {}),
                **({"inner_truncated": True} if tracker.inner_truncated else {}),
                **(
                    {"_q_star": tracker.info["q_star"]}
                    if tracker.action_requested and "q_star" in tracker.info
                    else {}
                ),
                **({"_terminated": True} if tracker.terminated else {}),
                **({"_truncated": True} if tracker.truncated else {}),
                **({"reward": tracker.reward} if tracker.reward != 0 else {}),
                **({"action_requested": True} if tracker.action_requested else {}),
            },
        )

env.close()
stores = [log.to_store() for log in logs]

## Save To The Hub

`push_stores_to_hub` uploads all datastores to one Hugging Face dataset repository. Each
datastore is saved as a named config, so downstream code can reload the same independent
streams with `load_stores_from_hub`.


In [ ]:
url = push_stores_to_hub(
    stores=stores, repo_id=DATASET_ID, split="train", private=False
)
print(f"Pushed to {url}")